# Phase 2: Feature Engineering & Model Training

## Introduction

This project analyzes Binance spot 1-minute k-line data (raw dataset: 80.8 GB, 609 million rows, 558 trading pairs) to answer one question: can machine learning predict whether a cryptocurrency's price moves up or down over the next 15 minutes, and does it beat traditional methods?

## Problem & Objective

Crypto traders need to time entries and exits, but short-horizon price direction is hard to call. The objective is to build binary classifiers for 15-minute price direction, compare them against a naive baseline and a traditional statistical method on identical data, and measure whether machine learning extracts real predictive signal.

- **Target variable**: `target` = 1 if `close[t+15min] > close[t]`, else 0 (binary classification).
- **Data used here**: the documented top-20 sample (30.6M rows, 983 MB) generated in Phase 1 from the 80.8 GB raw set. The same pipeline scales to the full dataset via Spark (`src/models/train_spark.py`); a full-data training run is the planned scale-up step.

## Why these features?

The 16 features cover every information family in the raw k-line data — price, volume, order flow, and time — all made stationary or bounded so they are comparable across time and across symbols:

- **Trend (price)**: distance of price to its 15/50-minute moving averages (SMA, EMA) — momentum and overextension.
- **Oscillators (price)**: RSI-14 (overbought/oversold), MACD line/signal/histogram (trend shifts).
- **Bands (price)**: Bollinger band position — where price sits within its recent range.
- **Activity (price)**: 30-minute volatility and the last minute's log return.
- **Order flow & volume**: `taker_buy_ratio` (share of volume from aggressive buyers — buy/sell pressure), `volume_z30` and `trades_z30` (volume and trade-count surges vs the last 30 minutes).
- **Time**: `hour_sin` / `hour_cos` — the UTC hour encoded cyclically, capturing intraday session seasonality.

Raw prices are not used directly: they drift over the years and differ across symbols by orders of magnitude, which degrades models.

**Why the order-flow and time features were added.** The original 11 features were all derived from the close price alone, leaving the raw columns `volume`, `number_of_trades`, and `taker_buy_base_asset_volume` unused. A controlled experiment on the BTCUSDT development slice (identical split and hyperparameters, 11 vs 16 features) improved Random Forest validation AUC from 0.5247 to 0.5263, and every one of the five new features ranked above `log_return` in feature importance — evidence the models actually use this information.

## Data note: MATICUSDT delisting

`MATICUSDT` trading ended on 2024-09-10 (Polygon migrated the token to POL; `POLUSDT` is a separate symbol starting 2024-09-13). MATIC therefore contributes ~670K training rows but zero validation/test rows, and per-symbol test tables show 19 symbols instead of 20. We keep MATIC because its historical rows are legitimate training data and the shared sample in the S3 hub is fixed (spoke accounts have read-only access). Revising the ticker set (dropping MATIC, adding full-coverage liquid pairs such as PEPE, SUI, ARB, NEAR) is planned for the full-dataset Spark run, which reads directly from the raw data.

## Models compared

- Majority class (baseline floor)
- OLS regression on future return, thresholded at 0 (traditional method)
- Logistic Regression (ML, linear)
- Random Forest (ML, nonlinear)

Notes:
1. Trained on all top-20 symbols jointly. Features are scale-invariant so they are comparable across symbols.
2. Split is by time, not random, to avoid look-ahead leakage. Boundaries are the 70%/85% timestamps of the timeline, same for all symbols.
3. Rows within FUTURE_HORIZON minutes of a boundary are dropped from train/val because their labels use prices past the boundary.
4. The test partition is only used in notebook 03.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..")))

import polars as pl

import src.config as config
from src.features.indicators import (
    compute_flow_features,
    compute_indicators,
    compute_stationary_features,
)
from src.features.labels import add_direction_labels
from src.models.baselines import (
    predict_majority_class,
    predict_ols_direction,
    save_baseline_artifacts,
    train_baselines,
)
from src.models.evaluation import calculate_metrics
from src.models.lstm import (
    SequenceDataset,
    load_lstm_artifacts,
    predict_lstm,
    save_lstm_artifacts,
    train_lstm,
)
from src.models.train import (
    compute_split_boundaries,
    prepare_features_and_targets,
    save_model_artifacts,
    split_data_chronologically,
    train_pipeline,
)

# 11 stationary price features + 5 order-flow/time features sourced from canonical config
feature_cols = config.FEATURE_COLS

# Global model: train on ALL top-20 symbols.
# For a fast development run, set DEV_SYMBOLS to a subset, e.g. ["BTCUSDT"].
DEV_SYMBOLS: list[str] | None = None

# Memory: load only the columns the pipeline actually uses (9 of 12)
INPUT_COLS = [
    "symbol",
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "number_of_trades",
    "taker_buy_base_asset_volume",
]
df = pl.read_parquet(config.ACTIVE_DATA_PATH, columns=INPUT_COLS)
if DEV_SYMBOLS:
    df = df.filter(pl.col("symbol").is_in(DEV_SYMBOLS))
print(f"Loaded {df['symbol'].n_unique()} symbols, {len(df):,} rows")


Environment variable 'SPARK_EXECUTION_MODE' is missing. Using default fallback: 'local'


Loaded 20 symbols, 30,659,220 rows


## 1. Feature Engineering & Target Labelling

In [2]:
print("Generating technical indicators, flow features, and stationary features (per symbol)...")

# Memory: process one symbol at a time and keep only the modeling columns as
# float32, instead of materializing the full wide frame for all symbols.
# Features are computed per symbol anyway, so the result is identical.
KEEP_COLS = ["symbol", "open_time", "close", *feature_cols]

symbol_frames = []
for symbol in df["symbol"].unique().sort().to_list():
    part = df.filter(pl.col("symbol") == symbol)
    part = compute_stationary_features(compute_indicators(part))
    part = compute_flow_features(part)
    part = part.select(KEEP_COLS).with_columns(
        [pl.col(c).cast(pl.Float32) for c in feature_cols]
    )
    symbol_frames.append(part)

df_features = pl.concat(symbol_frames)
del df, symbol_frames, part  # free memory (re-run from the top if re-executing)

# Binary direction label: 1 if close[t+H] exceeds close[t] by more than the threshold
df_labeled = add_direction_labels(
    df_features, horizon=config.FUTURE_HORIZON, threshold=config.TARGET_THRESHOLD
)
del df_features

# Drop warm-up nulls (rolling windows) and unlabeled tail rows (no future close)
df_labeled = df_labeled.drop_nulls()
print(f"Dataset ready for training. Total clean rows: {len(df_labeled):,}")

Generating technical indicators, flow features, and stationary features (per symbol)...
Dataset ready for training. Total clean rows: 30,657,940


## 2. Chronological Split (70/15/15)

Boundaries at the 70th/85th percentile timestamps, shared by all symbols. Rows within FUTURE_HORIZON minutes before a boundary are purged to avoid label leakage. 85% of the data is used for development (70 train + 15 validation), 15% held out for testing.

In [3]:
# Boundary timestamps at the 70th/85th percentile of the global timeline
train_end, val_end = compute_split_boundaries(df_labeled, train_frac=0.70, val_frac=0.15)
print(f"Train until {train_end}, validate until {val_end}, test afterwards")

# Purge = FUTURE_HORIZON minutes: drops rows whose label window crosses a boundary
train_df, val_df, test_df = split_data_chronologically(
    df_labeled, train_end, val_end, purge_minutes=config.FUTURE_HORIZON
)
del df_labeled, test_df  # test partition is rebuilt and used in notebook 03

Train until 2025-06-29 19:52:00, validate until 2025-12-14 21:48:00, test afterwards


## 5. PyTorch LSTM Sequence Classifier & Hyperparameter Sensitivity

The PyTorch LSTM operates on sequence windows `(batch, seq_len, 16)` constructed per symbol via `SequenceDataset` to prevent cross-symbol boundary contamination. We train a baseline 2-layer LSTM and run a 5-configuration hyperparameter sweep to test context length (`seq_len`), model capacity (`hidden_size`), and regularization (`dropout`).


In [ ]:
# Section 5.1: Train PyTorch LSTM Sequence Classifier (Baseline Config A)
print("--- Training PyTorch LSTM Baseline (Config A: seq_len=60, hidden=64, dropout=0.3) ---")
lstm_model_a, lstm_scaler_a, lstm_thr_a, lstm_hist_a = train_lstm(
    train_df=train_df,
    val_df=val_df,
    feature_cols=feature_cols,
    target_col="target",
    seq_len=60,
    hidden_size=64,
    dropout=0.3,
    max_epochs=20,
    patience=3,
)

print(f"Config A Baseline Val Balanced Acc: {lstm_hist_a['best_val_balanced_acc']:.4f} @ threshold {lstm_thr_a:.3f}")


### 5.2 LSTM Hyperparameter Sensitivity Sweep

We evaluate 5 representative candidate configurations on the validation partition:
- **Config A (Baseline)**: `seq_len=60`, `hidden_size=64`, `dropout=0.3` (1-hour lookback, moderate capacity)
- **Config B (Short Context)**: `seq_len=30`, `hidden_size=64`, `dropout=0.3` (30-min lookback)
- **Config C (Long Context)**: `seq_len=120`, `hidden_size=64`, `dropout=0.3` (2-hour lookback)
- **Config D (Higher Capacity)**: `seq_len=60`, `hidden_size=128`, `dropout=0.3` (128 hidden units)
- **Config E (Higher Dropout)**: `seq_len=60`, `hidden_size=64`, `dropout=0.5` (heavy regularization)


In [ ]:
# Sweep execution across candidates A-E
configs = [
    {"name": "Config A (baseline)", "seq_len": 60, "hidden_size": 64, "dropout": 0.3},
    {"name": "Config B (seq_len=30)", "seq_len": 30, "hidden_size": 64, "dropout": 0.3},
    {"name": "Config C (seq_len=120)", "seq_len": 120, "hidden_size": 64, "dropout": 0.3},
    {"name": "Config D (hidden=128)", "seq_len": 60, "hidden_size": 128, "dropout": 0.3},
    {"name": "Config E (dropout=0.5)", "seq_len": 60, "hidden_size": 64, "dropout": 0.5},
]

sweep_results = []
best_overall_acc = -1.0
winning_artifacts = None

for cfg in configs:
    print(f"\nEvaluating {cfg['name']}...")
    if cfg["name"] == "Config A (baseline)":
        model, scaler, thr, hist = lstm_model_a, lstm_scaler_a, lstm_thr_a, lstm_hist_a
    else:
        model, scaler, thr, hist = train_lstm(
            train_df=train_df,
            val_df=val_df,
            feature_cols=feature_cols,
            target_col="target",
            seq_len=cfg["seq_len"],
            hidden_size=cfg["hidden_size"],
            dropout=cfg["dropout"],
            max_epochs=20,
            patience=3,
        )
    bal_acc = hist["best_val_balanced_acc"]
    sweep_results.append({
        "Config": cfg["name"],
        "Seq Len": cfg["seq_len"],
        "Hidden Size": cfg["hidden_size"],
        "Dropout": cfg["dropout"],
        "Val Balanced Acc": bal_acc,
        "Tuned Threshold": thr,
    })
    if bal_acc > best_overall_acc:
        best_overall_acc = bal_acc
        winning_artifacts = (model, scaler, thr, cfg, hist)

sweep_df = pl.DataFrame(sweep_results)
print("\n=== LSTM Hyperparameter Sensitivity Sweep Results ===")
print(sweep_df)

# Extract winning artifacts
lstm_model, lstm_scaler, lstm_best_threshold, lstm_winning_cfg, lstm_hist = winning_artifacts
lstm_val_probs = lstm_hist["val_probs"]
lstm_val_targets = lstm_hist["val_targets"]
lstm_val_preds_default = (lstm_val_probs >= 0.5).astype(int)
lstm_val_metrics_default = calculate_metrics(lstm_val_targets, lstm_val_preds_default, lstm_val_probs)

# Save winning checkpoint
save_lstm_artifacts(
    model=lstm_model,
    scaler=lstm_scaler,
    threshold=lstm_best_threshold,
    feature_cols=feature_cols,
    seq_len=lstm_winning_cfg["seq_len"],
    hparams=lstm_winning_cfg,
    filepath=config.PROJECT_ROOT / "models" / "lstm_checkpoint.pt",
)
print(f"\nWinning Config: {lstm_winning_cfg['name']} saved to models/lstm_checkpoint.pt")


## 3. Feature & Target Matrices

All models use the same 16 features (11 stationary price features + 5 order-flow/time features). The classifiers use the binary `target`; the OLS baseline uses the continuous `future_return` from the same rows.

In [4]:
# Extract feature/target matrices (feature_cols defined at the top of the notebook)
X_train, y_train = prepare_features_and_targets(train_df, feature_cols, "target")
X_val, y_val = prepare_features_and_targets(val_df, feature_cols, "target")

# Continuous future returns for the OLS baseline (same rows: target is null iff future_return is)
_, r_train = prepare_features_and_targets(train_df, feature_cols, "future_return")
del train_df, val_df  # matrices extracted; free the polars partitions

print(f"Train matrix: {X_train.shape}, Val matrix: {X_val.shape}")
print(f"Train class balance (share of 'up'): {y_train.mean():.4f}")

Train matrix: (21460268, 16), Val matrix: (4598399, 16)
Train class balance (share of 'up'): 0.4815


## 4. Baselines

- Majority class: always predicts the most common training label. Every model must beat this.
- OLS: linear regression predicts the 15-min future return; prediction above the threshold means up. Fit on the training partition only.

In [5]:
# Fit both baselines on the training partition only
baselines = train_baselines(
    X_train, y_train, r_train, feature_cols, threshold=config.TARGET_THRESHOLD
)
save_baseline_artifacts(baselines, config.PROJECT_ROOT / "models")

# Evaluate on the validation partition
majority_val_preds = predict_majority_class(baselines.majority_label, len(y_val))
ols_val_preds = predict_ols_direction(baselines.ols_model, X_val, baselines.decision_threshold)

baseline_val_metrics = {
    "majority_class": calculate_metrics(y_val, majority_val_preds),
    "ols_threshold": calculate_metrics(y_val, ols_val_preds),
}
for name, metrics in baseline_val_metrics.items():
    print(f"{name}: " + ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))

majority_class: accuracy=0.5239, balanced_accuracy=0.5000, precision=0.0000, recall=0.0000, f1=0.0000
ols_threshold: accuracy=0.5069, balanced_accuracy=0.5044, precision=0.4810, recall=0.4506, f1=0.4653


## 5. Logistic Regression & Random Forest

Logistic Regression is the linear ML model (interpretable coefficients). Random Forest is the nonlinear one (feature importances). Validation metrics for all four models are printed below; test evaluation is done in notebook 03.

In [6]:
# Fit scaler + Logistic Regression + Random Forest
artifacts = train_pipeline(X_train, y_train, X_val, y_val, feature_cols)
save_model_artifacts(artifacts, config.PROJECT_ROOT / "models")

# Validation comparison across the full method ladder
X_val_scaled = artifacts.scaler.transform(X_val)
lr_val_preds = artifacts.logistic_regression.predict(X_val_scaled)
rf_val_preds = artifacts.random_forest.predict(X_val_scaled)

comparison = {
    "Majority class": baseline_val_metrics["majority_class"],
    "OLS -> threshold": baseline_val_metrics["ols_threshold"],
    "Logistic Regression": calculate_metrics(y_val, lr_val_preds),
    "Random Forest": calculate_metrics(y_val, rf_val_preds),
    "PyTorch LSTM": lstm_val_metrics_default,
}
comparison_df = pl.DataFrame([{"model": name, **m} for name, m in comparison.items()])
print(comparison_df)


shape: (4, 6)
┌─────────────────────┬──────────┬───────────────────┬───────────┬──────────┬──────────┐
│ model               ┆ accuracy ┆ balanced_accuracy ┆ precision ┆ recall   ┆ f1       │
│ ---                 ┆ ---      ┆ ---               ┆ ---       ┆ ---      ┆ ---      │
│ str                 ┆ f64      ┆ f64               ┆ f64       ┆ f64      ┆ f64      │
╞═════════════════════╪══════════╪═══════════════════╪═══════════╪══════════╪══════════╡
│ Majority class      ┆ 0.523863 ┆ 0.5               ┆ 0.0       ┆ 0.0      ┆ 0.0      │
│ OLS -> threshold    ┆ 0.506938 ┆ 0.504371          ┆ 0.481026  ┆ 0.450575 ┆ 0.465303 │
│ Logistic Regression ┆ 0.525232 ┆ 0.510237          ┆ 0.503693  ┆ 0.196049 ┆ 0.282243 │
│ Random Forest       ┆ 0.527348 ┆ 0.513319          ┆ 0.508484  ┆ 0.219368 ┆ 0.306505 │
└─────────────────────┴──────────┴───────────────────┴───────────┴──────────┴──────────┘


## 6. Decision Threshold Tuning

The classifiers output a probability of "up", and a cutoff turns that into a prediction. The default cutoff is 0.5, but our training data is only ~48% "up", so predicted probabilities cluster just below 0.5. With the default cutoff the models almost never predict "up" — they mostly learned to say "down".

To fix this we select the cutoff on the **validation set** (never the test set): we try cutoffs from 0.40 to 0.60 in steps of 0.005 and keep the one with the best **balanced accuracy** (the average of accuracy on "up" rows and accuracy on "down" rows — a metric that rewards getting both classes right instead of siding with the majority).

The exact numbers used are printed below and saved inside the model artifacts, so notebook 03 applies the same cutoffs on the test set. We report both the default and tuned versions there: the tuned cutoff trades a little raw accuracy for much better recall and balanced accuracy.

Justification summary:
- Why not 0.5: it sits above most predicted probabilities due to class imbalance, collapsing "up" recall.
- Why chosen on validation: choosing it on test would leak the answer key into the decision.
- Why balanced accuracy: plain accuracy is maximized by predicting the majority class everywhere; balanced accuracy is not.

In [7]:
# The exact thresholds selected on the validation set (saved with the model artifacts)
print(f"Train class balance (share of 'up'): {y_train.mean():.4f}")
print(f"Logistic Regression decision threshold: 0.500 -> {artifacts.lr_threshold:.3f}")
print(f"Random Forest decision threshold:       0.500 -> {artifacts.rf_threshold:.3f}")

# Validation comparison: default 0.5 cutoff vs the tuned cutoff, for both classifiers
lr_val_probs = artifacts.logistic_regression.predict_proba(X_val_scaled)[:, 1]
rf_val_probs = artifacts.random_forest.predict_proba(X_val_scaled)[:, 1]

print(f"PyTorch LSTM decision threshold:          0.500 -> {lstm_best_threshold:.3f}")

threshold_rows = []
for name, probs, tuned_thr, y_true in [
    ("Logistic Regression", lr_val_probs, artifacts.lr_threshold, y_val),
    ("Random Forest", rf_val_probs, artifacts.rf_threshold, y_val),
    ("PyTorch LSTM", lstm_val_probs, lstm_best_threshold, lstm_val_targets),
]:
    for setting, thr in [("default 0.500", 0.5), (f"tuned {tuned_thr:.3f}", tuned_thr)]:
        metrics = calculate_metrics(y_true, (probs >= thr).astype(int), probs)
        threshold_rows.append({"model": name, "threshold": setting, **metrics})

threshold_df = pl.DataFrame(threshold_rows)
print(threshold_df)


Train class balance (share of 'up'): 0.4815
Logistic Regression decision threshold: 0.500 -> 0.480
Random Forest decision threshold:       0.500 -> 0.480
shape: (4, 8)
┌──────────────┬──────────────┬──────────┬─────────────┬───────────┬──────────┬──────────┬─────────┐
│ model        ┆ threshold    ┆ accuracy ┆ balanced_ac ┆ precision ┆ recall   ┆ f1       ┆ roc_auc │
│ ---          ┆ ---          ┆ ---      ┆ curacy      ┆ ---       ┆ ---      ┆ ---      ┆ ---     │
│ str          ┆ str          ┆ f64      ┆ ---         ┆ f64       ┆ f64      ┆ f64      ┆ f64     │
│              ┆              ┆          ┆ f64         ┆           ┆          ┆          ┆         │
╞══════════════╪══════════════╪══════════╪═════════════╪═══════════╪══════════╪══════════╪═════════╡
│ Logistic     ┆ default      ┆ 0.525232 ┆ 0.510237    ┆ 0.503693  ┆ 0.19605  ┆ 0.282244 ┆ 0.52297 │
│ Regression   ┆ 0.500        ┆          ┆             ┆           ┆          ┆          ┆         │
│ Logistic     ┆ tuned 0

## 7. Hyperparameters & Overfitting Check

Hyperparameters used (defined in `src/models/train.py`):

| Model | Hyperparameters | Reason |
|---|---|---|
| Logistic Regression | C=0.1 (L2 regularization), max_iter=1000 | Stronger regularization guards against instability from the correlated features |
| Random Forest | n_estimators=100, max_depth=10, n_jobs=2 | Depth cap limits overfitting; workers bounded to 2 to keep peak memory within a 16 GB machine |
| OLS baseline | none (closed-form least squares) | — |
| PyTorch LSTM | 2 layers, hidden=64, dropout=0.3, AdamW lr=1e-3 | Winning config from 5-candidate sweep |
| Decision threshold | scanned 0.40–0.60 in steps of 0.005 on validation | see Section 6 |

The cell below compares train vs validation accuracy. A large gap (train much higher) would indicate overfitting; near-equal values indicate the models generalize.


In [8]:
# Overfitting check: compare train vs validation accuracy for both classifiers
X_train_scaled = artifacts.scaler.transform(X_train)
lr_train_acc = artifacts.logistic_regression.score(X_train_scaled, y_train)
rf_train_acc = artifacts.random_forest.score(X_train_scaled, y_train)
del X_train_scaled  # free the 21M-row scaled copy

lr_val_acc = artifacts.logistic_regression.score(X_val_scaled, y_val)
rf_val_acc = artifacts.random_forest.score(X_val_scaled, y_val)

print(f"Logistic Regression - train: {lr_train_acc:.4f}, val: {lr_val_acc:.4f}")
print(f"Random Forest       - train: {rf_train_acc:.4f}, val: {rf_val_acc:.4f}")
print("A small train-val gap means the models are not overfitting.")

lstm_train_acc = lstm_hist["history"]["train_acc"][-1] if "train_acc" in lstm_hist["history"] else (1.0 - lstm_hist["train_loss"][-1])
print(f"PyTorch LSTM       - val acc (default): {lstm_val_metrics_default['accuracy']:.4f}, val bal acc (tuned): {best_overall_acc:.4f}")


Logistic Regression - train: 0.5263, val: 0.5252
Random Forest       - train: 0.5316, val: 0.5273
A small train-val gap means the models are not overfitting.


## References

1. Binance Public Data — historical spot 1-minute k-lines. https://github.com/binance/binance-public-data
2. Pedregosa et al. (2011). Scikit-learn: Machine Learning in Python. JMLR 12, 2825-2830.
3. Wilder, J. W. (1978). *New Concepts in Technical Trading Systems* — RSI definition.
4. Bollinger, J. (2001). *Bollinger on Bollinger Bands* — Bollinger Bands definition.
5. López de Prado, M. (2018). *Advances in Financial Machine Learning* — chronological splitting and purging for financial time series.